# JsonCleaningScript

This analysis script is used in Hew's replication of Dang et al. (2025). CSV files from each participant were saved to my Google Drive via Data Pipe after the completion of the jsPsych study. The goal is to take each `.csv` file and convert it to the tidy wide data format that we need. Each .csv that is outputted by jsPsych contains entries that are Jsons that we need to unravel to actually get the data that we need.

Claude code Sonnet 5.5 Medium (2026, Nov 17) was used to assist in coding this.

```
Before you begin, make sure that the file structure looks like:
.
├── Data
│   ├── 2324820432840.csv
│   └── 2482394023894.csv
└── JsonCleaning.ipynb
```

# Script

In [ ]:
import pandas as pd
import json
from pathlib import Path # This is a built-in Python feature

# Path() creates an object representing a file directory to allow us to use object oriented methods
# to interact with the files. Path("Data") is selecting the "Data" folder that we have.
# glob() takes Unix style pathname pattern.

DataFiles = Path("Data").glob("*.csv")
LongRows = []

for file in DataFiles:
    df = pd.read_csv(file)
    # Because this column is uniform for each file, it does not matter which .iloc[] you choose as long as it is in range
    ID = df['subject_id'].iloc[0]

    # Grab the last time elpased (total time in Experiment)
    TotalTime = df['time_elapsed'].iloc[-1]
    TotalMinutes = (TotalTime/1000)/60

    LongRows.append({
        "ID":ID,
        "TrialIndex" : "X",
        "item" : "TotalMinutes",
        "value" : TotalMinutes
    })
    for i, row in df.iterrows():
        # Keep looping through the responses until the response col 
        Response = row['response']
        if pd.isna(Response) or not str(Response).strip().startswith("{"):
            continue
        Parsed = json.loads(Response)
        if not isinstance(Parsed, dict):
            Parsed = {"response":Parsed}
        
        for key, value in Parsed.items():
            LongRows.append({
                "ID": ID,
                "TrialIndex": row["trial_index"],
                "item": key,
                "value": value,
            })
LongDataFrame = pd.DataFrame(LongRows)

# Create Temporary Column Names
LongDataFrame["Col"] = LongDataFrame["TrialIndex"].astype(str) + "_" + LongDataFrame["item"]


# Pivot the data so that each ID gets a row, each unique value in Col becomes a column, and use the "value" column in the dataframe to fill the cell.
WideDataFrame = LongDataFrame.pivot(index = "ID", columns = "Col", values = "value").reset_index()
WideDataFrame

# Export the Data to CSV

In [ ]:
# WideDataFrame.to_csv("NOT_FOR_PUBLIC_TidyData.csv", index=True)

# Remove freeform responses for participant privacy
del WideDataFrame['7_GratitudeResponse']
del WideDataFrame['7_EventResponse']
WideDataFrame.to_csv("CleanDataPublic.csv", index=True)
display(WideDataFrame)